In [2]:
import numpy as np

# Structured historical price series (representing daily closing prices from Yahoo Finance)
recent_prices = [
    218.5, 219.2, 217.8, 220.1, 221.5, 220.8, 222.4, 223.1, 221.9, 223.5,
    224.2, 222.8, 223.9, 225.1, 224.6, 226.0, 225.5, 227.1, 226.4, 228.0,
    227.2, 228.5, 229.1, 228.4, 230.2, 229.8, 231.5, 230.9, 232.0, 231.4
]

class StockRiskMachine:
    def __init__(self, historical_prices, max_allowed_drawdown_pct=3.0):
        self.prices = historical_prices
        self.max_drawdown = max_allowed_drawdown_pct
        self.current_price = self.prices[-1]
        
    def simulate_future_scenarios(self, proposed_trade_size, price_shocks=[-0.02, 0.0, 0.02]):
        # Calculate historical volatility from price series
        returns = np.diff(self.prices) / self.prices[:-1]
        volatility = np.std(returns)
        
        predicted_worlds = []
        for shock in price_shocks:
            # Project future price under different scenario branches (stress testing)
            simulated_price = self.current_price * (1.0 + shock + (volatility * np.sign(shock)))
            drawdown_pct = ((self.current_price - simulated_price) / self.current_price) * 100
            
            world = {
                "scenario": shock,
                "projected_price": simulated_price,
                "drawdown": max(0.0, drawdown_pct), # Track downside drawdowns
                "exposure": proposed_trade_size * simulated_price
            }
            predicted_worlds.append(world)
        return predicted_worlds

    def verify_safety_invariant(self, predicted_worlds):
        # Box P: In ALL possible future worlds, drawdown must NOT exceed max_allowed_drawdown
        for world in predicted_worlds:
            if world["drawdown"] > self.max_drawdown:
                return False, f"Invariant Violation! Shock branch triggers drawdown of {world['drawdown']:.2f}% (Limit: {self.max_drawdown}%)"
        return True, "All projected futures satisfy safety invariant."

    def evaluate_trade(self, trade_size):
        scenarios = self.simulate_future_scenarios(trade_size)
        is_safe, message = self.verify_safety_invariant(scenarios)
        
        if is_safe:
            return f"APPROVED: Position size {trade_size} units is safe across all projected market states."
        else:
            return f"BLOCKED: {message}"

# Initialize machine with market price data and a 3.0% maximum risk limit
machine = StockRiskMachine(recent_prices, max_allowed_drawdown_pct=3.0)

print("--- Test: Evaluating Proposed Trade ---")
print(machine.evaluate_trade(100))

--- Test: Evaluating Proposed Trade ---
APPROVED: Position size 100 units is safe across all projected market states.
